<a href="https://colab.research.google.com/github/FatemehNMT/Visual-SLAM-Book-Google-Colab/blob/main/Chapter_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Replace "CV_GRAY2BGR" with "cv::COLOR_BGR2RGB"

# **Mount the Drive**

In [ ]:
%cd /content

/content


In [ ]:
# Mount the Drive to use tha data for testing the library

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **installation**

## Install PCL

In [ ]:
%cd /content/

/content


In [ ]:
!ls

drive  sample_data


In [ ]:
!sudo apt-get install libpcl-dev pcl-tools

In [ ]:
!ls

drive  sample_data


## Install Eigen

In [ ]:
!sudo apt install libeigen3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libeigen3-dev is already the newest version (3.4.0-2ubuntu2).
libeigen3-dev set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [ ]:
%cd /content/

/content


In [ ]:
!git clone https://gitlab.com/libeigen/eigen.git

%cd eigen

!mkdir build
%cd build

!cmake ..

!sudo make install

## Install Ceres

In [ ]:
%cd /content
!mkdir Ceres
%cd Ceres

# google-glog + gflags
!sudo apt-get install libgoogle-glog-dev libgflags-dev

# Use ATLAS for BLAS & LAPACK
!sudo apt-get install libatlas-base-dev

# Eigen3
!sudo apt-get install libeigen3-dev

# SuiteSparse (optional)
!sudo apt-get install libsuitesparse-dev

!sudo apt-get install liblapack-dev libsuitesparse-dev libcxsparse3 libgflags-dev libgoogle-glog-dev libgtest-dev

!git clone https://github.com/ceres-solver/ceres-solver.git

!sudo apt-get install libceres-dev

!tar -xzf "/content/drive/MyDrive/slambook_data/ch10/ceres-solver-2.1.0.tar.gz" -C "/content/Ceres/ceres-solver"

%cd ceres-solver

!mkdir ceres-bin
%cd ceres-bin

!cmake ../ceres-solver-2.1.0/

!make -j8

!make install

## Install Sophus

In [ ]:
%cd /content/

/content


In [ ]:
!git clone https://github.com/strasdat/Sophus.git

%cd Sophus/

!git clone https://github.com/Microsoft/vcpkg.git
%cd vcpkg

!./bootstrap-vcpkg.sh

!./vcpkg integrate install

!./vcpkg install sophus

# %cd ../

# !mkdir build
# %cd build

# !cmake ..

# !make

%cd /content

# **ch12/dense_monocular/dense_mapping.cpp**

## PDF Page 268

In [ ]:
%cd /content/

/content


In [ ]:
!rm -r MyExample_dense
!mkdir MyExample_dense
%cd /content/MyExample_dense

/content/MyExample_dense


In [ ]:
%%writefile CMakeLists.txt

cmake_minimum_required(VERSION 3.1)
project(ch8)

set( CMAKE_BUILD_TYPE "Release" )
set( CMAKE_CXX_FLAGS "-std=c++17 -O3" )
add_definitions("-DENABLE_SSE")
# set(CMAKE_CXX_FLAGS "-std=c++17 ${SSE_FLAGS} -g -O3 -march=native")

############### dependencies ######################
# OpenCV
find_package(OpenCV REQUIRED)
include_directories(${OpenCV_INCLUDE_DIRS})
include_directories("/usr/include/opencv4/opencv2")
link_directories("/usr/local/lib")
include_directories("/usr/local/lib")

# Eigen
include_directories("/content/eigen")
include_directories("/usr/include/eigen3")
find_package(Eigen3 REQUIRED)

# Sophus
include_directories("/content/Sophus")

add_executable(dense_mapping dense_mapping.cpp)
target_link_libraries(dense_mapping ${THIRD_PARTY_LIBS})

Writing CMakeLists.txt


In [ ]:
%%writefile dense_mapping.cpp

#include <iostream>
#include <vector>
#include <fstream>
#include <optional>
using namespace std;

#include <boost/timer.hpp>

// for sophus
#include <sophus/se3.hpp>

using Sophus::SE3d;

// for eigen
#include <Eigen/Core>
#include <Eigen/Geometry>

using namespace Eigen;

#include <opencv2/core/core.hpp>
#include <opencv2/highgui/highgui.hpp>
#include <opencv2/imgproc/imgproc.hpp>

using namespace cv;

/**********************************************
* This program demonstrates dense depth estimation of a monocular camera under a known trajectory
* Uses epipolar search + NCC matching, corresponding to Section 12.2 of the book
* Please note that this program is not perfect, you can definitely improve it - I am actually deliberately exposing some problems (this is an excuse).
***********************************************/

// ------------------------------------------------------------------
// parameters
const int boarder = 20;         // Edge width
const int width = 640; // Image width
const int height = 480; // Image height
const double fx = 481.2f; // Camera internal parameters
const double fy = -480.0f;
const double cx = 319.5f;
const double cy = 239.5f;
const int ncc_window_size = 3;    // Half-width of the window taken by NCC
const int ncc_area = (2 * ncc_window_size + 1) * (2 * ncc_window_size + 1); // NCC window area
const double min_cov = 0.1; // Convergence judgment: minimum variance
const double max_cov = 10; // Divergence judgment: maximum variance

// ------------------------------------------------------------------

// Important functions
/// Read data from the REMODE dataset
bool readDatasetFiles(
    const string &path,
    vector<string> &color_image_files,
    vector<SE3d> &poses,
    cv::Mat &ref_depth
);

/**
* Update the depth estimate based on the new image
* @param ...... ref reference image
* @param ...... curr current image
* @param ...... T_C_R reference image to current image pose
* @param ...... depth depth
* @param ...... depth_cov depth variance
* @return ......  success or failure
 */
bool update(
    const Mat &ref,
    const Mat &curr,
    const SE3d &T_C_R,
    Mat &depth,
    Mat &depth_cov2
);

/**
* Epipolar search
* @param ...... ref reference image
* @param ......  curr current image
* @param ......  T_C_R pose
* @param ......  pt_ref position of the point in the reference image
* @param ......  depth_mu depth mean
* @param ......  depth_cov depth variance
* @param ......  pt_curr current point
* @param ......  epipolar_direction epipolar direction
* @return ...... success or failure
 */
bool epipolarSearch(
    const Mat &ref,
    const Mat &curr,
    const SE3d &T_C_R,
    const Vector2d &pt_ref,
    const double &depth_mu,
    const double &depth_cov,
    Vector2d &pt_curr,
    Vector2d &epipolar_direction
);

/**
 * Update depth filter
* @param pt_ref  ...... reference image point
* @param pt_curr  ...... current image point
* @param T_C_R  ...... pose
* @param epipolar_direction  ...... epipolar direction
* @param depth  ...... depth mean
* @param depth_cov2  ...... depth direction
* @return ...... success
 */
bool updateDepthFilter(
    const Vector2d &pt_ref,
    const Vector2d &pt_curr,
    const SE3d &T_C_R,
    const Vector2d &epipolar_direction,
    Mat &depth,
    Mat &depth_cov2
);

/**
* Calculate NCC score
* @param ref ...... reference image
* @param curr ...... current image
* @param pt_ref ...... reference point
* @param pt_curr ...... current point
* @return ...... NCC score
 */
double NCC(const Mat &ref, const Mat &curr, const Vector2d &pt_ref, const Vector2d &pt_curr);

// Bilinear grayscale interpolation
inline double getBilinearInterpolatedValue(const Mat &img, const Vector2d &pt) {
    uchar *d = &img.data[int(pt(1, 0)) * img.step + int(pt(0, 0))];
    double xx = pt(0, 0) - floor(pt(0, 0));
    double yy = pt(1, 0) - floor(pt(1, 0));
    return ((1 - xx) * (1 - yy) * double(d[0]) +
            xx * (1 - yy) * double(d[1]) +
            (1 - xx) * yy * double(d[img.step]) +
            xx * yy * double(d[img.step + 1])) / 255.0;
}

// ------------------------------------------------------------------
//Some widgets
// Display estimated depth map
void plotDepth(const Mat &depth_truth, const Mat &depth_estimate);

// Pixel to camera coordinate system
inline Vector3d px2cam(const Vector2d px) {
    return Vector3d(
        (px(0, 0) - cx) / fx,
        (px(1, 0) - cy) / fy,
        1
    );
}

// Camera coordinates to pixels
inline Vector2d cam2px(const Vector3d p_cam) {
    return Vector2d(
        p_cam(0, 0) * fx / p_cam(2, 0) + cx,
        p_cam(1, 0) * fy / p_cam(2, 0) + cy
    );
}

// Check if a point is within the image border
inline bool inside(const Vector2d &pt) {
    return pt(0, 0) >= boarder && pt(1, 0) >= boarder
           && pt(0, 0) + boarder < width && pt(1, 0) + boarder <= height;
}

// Show Epipolar Matching
void showEpipolarMatch(const Mat &ref, const Mat &curr, const Vector2d &px_ref, const Vector2d &px_curr);

// Show Polar Lines
void showEpipolarLine(const Mat &ref, const Mat &curr, const Vector2d &px_ref, const Vector2d &px_min_curr,
                      const Vector2d &px_max_curr);

/// Evaluating Depth Estimation
void evaludateDepth(const Mat &depth_truth, const Mat &depth_estimate);
// ------------------------------------------------------------------


int main(int argc, char **argv) {
    if (argc != 2) {
        cout << "Usage: dense_mapping path_to_test_dataset" << endl;
        return -1;
    }

    // Reading data from a dataset
    vector<string> color_image_files;
    vector<SE3d> poses_TWC;
    Mat ref_depth;
    bool ret = readDatasetFiles(argv[1], color_image_files, poses_TWC, ref_depth);
    if (ret == false) {
        cout << "Reading image files failed!" << endl;
        return -1;
    }
    cout << "read total " << color_image_files.size() << " files." << endl;

    // First picture
    Mat ref = imread(color_image_files[0], 0);                // gray-scale image
    SE3d pose_ref_TWC = poses_TWC[0];
    double init_depth = 3.0;    // Depth initial value
    double init_cov2 = 3.0; // Variance initial value
    Mat depth(height, width, CV_64F, init_depth);             // Depth Map
    Mat depth_cov2(height, width, CV_64F, init_cov2);         // Depth map variance

    for (int index = 1; index < color_image_files.size(); index++) {
        cout << "*** loop " << index << " ***" << endl;
        Mat curr = imread(color_image_files[index], 0);
        if (curr.data == nullptr) continue;
        SE3d pose_curr_TWC = poses_TWC[index];
        SE3d pose_T_C_R = pose_curr_TWC.inverse() * pose_ref_TWC;   // Coordinate transformation relationship: T_C_W * T_W_R = T_C_R
        update(ref, curr, pose_T_C_R, depth, depth_cov2);
        evaludateDepth(ref_depth, depth);
        plotDepth(ref_depth, depth);
        // imshow("image", curr);
        imwrite("image.png", curr);

        waitKey(1);
    }

    cout << "estimation returns, saving depth map ..." << endl;
    imwrite("depth.png", depth);
    cout << "done." << endl;

    return 0;
}

bool readDatasetFiles(
    const string &path,
    vector<string> &color_image_files,
    std::vector<SE3d> &poses,
    cv::Mat &ref_depth) {
    ifstream fin(path + "/first_200_frames_traj_over_table_input_sequence.txt");
    if (!fin) return false;

    while (!fin.eof()) {
        // Data format: Image file name tx, ty, tz, qx, qy, qz, qw, note that it is TWC instead of TCW
        string image;
        fin >> image;
        double data[7];
        for (double &d:data) fin >> d;

        color_image_files.push_back(path + string("/images/") + image);
        poses.push_back(
            SE3d(Quaterniond(data[6], data[3], data[4], data[5]),
                 Vector3d(data[0], data[1], data[2]))
        );
        if (!fin.good()) break;
    }
    fin.close();

    // load reference depth
    fin.open(path + "/depthmaps/scene_000.depth");
    ref_depth = cv::Mat(height, width, CV_64F);
    if (!fin) return false;
    for (int y = 0; y < height; y++)
        for (int x = 0; x < width; x++) {
            double depth = 0;
            fin >> depth;
            ref_depth.ptr<double>(y)[x] = depth / 100.0;
        }

    return true;
}

// Update the entire depth map
bool update(const Mat &ref, const Mat &curr, const SE3d &T_C_R, Mat &depth, Mat &depth_cov2) {
    for (int x = boarder; x < width - boarder; x++)
        for (int y = boarder; y < height - boarder; y++) {
            // Iterate over each pixel
            if (depth_cov2.ptr<double>(y)[x] < min_cov || depth_cov2.ptr<double>(y)[x] > max_cov) // Depth has converged or diverged
                continue;
            // Search for a match of (x,y) on an epipolar line
            Vector2d pt_curr;
            Vector2d epipolar_direction;
            bool ret = epipolarSearch(
                ref,
                curr,
                T_C_R,
                Vector2d(x, y),
                depth.ptr<double>(y)[x],
                sqrt(depth_cov2.ptr<double>(y)[x]),
                pt_curr,
                epipolar_direction
            );

            if (ret == false) // Matching failed
                continue;

            // Uncomment this to show the matches
            // showEpipolarMatch(ref, curr, Vector2d(x, y), pt_curr);

            // Matching successful, update depth map
            updateDepthFilter(Vector2d(x, y), pt_curr, T_C_R, epipolar_direction, depth, depth_cov2);
        }
}

// Polar line search
// For methods, see Sections 12.2 and 12.3
bool epipolarSearch(
    const Mat &ref, const Mat &curr,
    const SE3d &T_C_R, const Vector2d &pt_ref,
    const double &depth_mu, const double &depth_cov,
    Vector2d &pt_curr, Vector2d &epipolar_direction) {
    Vector3d f_ref = px2cam(pt_ref);
    f_ref.normalize();
    Vector3d P_ref = f_ref * depth_mu;    // P vector of the reference frame

    Vector2d px_mean_curr = cam2px(T_C_R * P_ref); // Pixels projected by depth mean
    double d_min = depth_mu - 3 * depth_cov, d_max = depth_mu + 3 * depth_cov;
    if (d_min < 0.1) d_min = 0.1;
    Vector2d px_min_curr = cam2px(T_C_R * (f_ref * d_min));    //Pixels projected at minimum depth
    Vector2d px_max_curr = cam2px(T_C_R * (f_ref * d_max)); // Pixels projected at maximum depth

    Vector2d epipolar_line = px_max_curr - px_min_curr;    // Polar line (line segment form)
    epipolar_direction = epipolar_line;        // Polar direction
    epipolar_direction.normalize();
    double half_length = 0.5 * epipolar_line.norm();    // Half length of the polar segment
    if (half_length > 100) half_length = 100;   // We don't want to search too much.

    // Uncomment this sentence to display the polar lines (line segments)
    // showEpipolarLine( ref, curr, pt_ref, px_min_curr, px_max_curr );

    // Search on the extreme line, with the depth mean point as the center, and take half the length on each side
    double best_ncc = -1.0;
    Vector2d best_px_curr;
    for (double l = -half_length; l <= half_length; l += 0.7) { // l+=sqrt(2)
        Vector2d px_curr = px_mean_curr + l * epipolar_direction;  // Matchup allocation
        if (!inside(px_curr))
            continue;
        // Calculate the matching point and the reference frame NCC
        double ncc = NCC(ref, curr, pt_ref, px_curr);
        if (ncc > best_ncc) {
            best_ncc = ncc;
            best_px_curr = px_curr;
        }
    }
    if (best_ncc < 0.85f)      // Only trust matches with a high NCC
        return false;
    pt_curr = best_px_curr;
    return true;
}

double NCC(
    const Mat &ref, const Mat &curr,
    const Vector2d &pt_ref, const Vector2d &pt_curr) {
    // Zero mean - normalized cross correlation
    // Calculate the mean first
    double mean_ref = 0, mean_curr = 0;
    vector<double> values_ref, values_curr; // The average of the reference frame and the current frame
    for (int x = -ncc_window_size; x <= ncc_window_size; x++)
        for (int y = -ncc_window_size; y <= ncc_window_size; y++) {
            double value_ref = double(ref.ptr<uchar>(int(y + pt_ref(1, 0)))[int(x + pt_ref(0, 0))]) / 255.0;
            mean_ref += value_ref;

            double value_curr = getBilinearInterpolatedValue(curr, pt_curr + Vector2d(x, y));
            mean_curr += value_curr;

            values_ref.push_back(value_ref);
            values_curr.push_back(value_curr);
        }

    mean_ref /= ncc_area;
    mean_curr /= ncc_area;

    // calculate Zero mean NCC
    double numerator = 0, demoniator1 = 0, demoniator2 = 0;
    for (int i = 0; i < values_ref.size(); i++) {
        double n = (values_ref[i] - mean_ref) * (values_curr[i] - mean_curr);
        numerator += n;
        demoniator1 += (values_ref[i] - mean_ref) * (values_ref[i] - mean_ref);
        demoniator2 += (values_curr[i] - mean_curr) * (values_curr[i] - mean_curr);
    }
    return numerator / sqrt(demoniator1 * demoniator2 + 1e-10);   // Prevent zero in the denominator
}

bool updateDepthFilter(
    const Vector2d &pt_ref,
    const Vector2d &pt_curr,
    const SE3d &T_C_R,
    const Vector2d &epipolar_direction,
    Mat &depth,
    Mat &depth_cov2) {
    // I don't know if anyone is watching this part
    // Use triangulation to calculate depth
    SE3d T_R_C = T_C_R.inverse();
    Vector3d f_ref = px2cam(pt_ref);
    f_ref.normalize();
    Vector3d f_curr = px2cam(pt_curr);
    f_curr.normalize();

    // Equation
    // d_ref * f_ref = d_cur * ( R_RC * f_cur ) + t_RC
    // f2 = R_RC * f_cur
    // Transformed into the following matrix equation system
    // => [ f_ref^T f_ref, -f_ref^T f2 ] [d_ref]   [f_ref^T t]
    //    [ f_2^T f_ref, -f2^T f2      ] [d_cur] = [f2^T t   ]
    Vector3d t = T_R_C.translation();
    Vector3d f2 = T_R_C.so3() * f_curr;
    Vector2d b = Vector2d(t.dot(f_ref), t.dot(f2));
    Matrix2d A;
    A(0, 0) = f_ref.dot(f_ref);
    A(0, 1) = -f_ref.dot(f2);
    A(1, 0) = -A(0, 1);
    A(1, 1) = -f2.dot(f2);
    Vector2d ans = A.inverse() * b;
    Vector3d xm = ans[0] * f_ref;           // ref Side Results
    Vector3d xn = t + ans[1] * f2;          // cur result
    Vector3d p_esti = (xm + xn) / 2.0;      // The position of P is taken as the average of the two
    double depth_estimation = p_esti.norm();   // Depth value

    // Calculate uncertainty (one pixel error)
    Vector3d p = f_ref * depth_estimation;
    Vector3d a = p - t;
    double t_norm = t.norm();
    double a_norm = a.norm();
    double alpha = acos(f_ref.dot(t) / t_norm);
    double beta = acos(-a.dot(t) / (a_norm * t_norm));
    Vector3d f_curr_prime = px2cam(pt_curr + epipolar_direction);
    f_curr_prime.normalize();
    double beta_prime = acos(f_curr_prime.dot(-t) / t_norm);
    double gamma = M_PI - alpha - beta_prime;
    double p_prime = t_norm * sin(beta_prime) / sin(gamma);
    double d_cov = p_prime - depth_estimation;
    double d_cov2 = d_cov * d_cov;

    // Gaussian Fusion
    double mu = depth.ptr<double>(int(pt_ref(1, 0)))[int(pt_ref(0, 0))];
    double sigma2 = depth_cov2.ptr<double>(int(pt_ref(1, 0)))[int(pt_ref(0, 0))];

    double mu_fuse = (d_cov2 * mu + sigma2 * depth_estimation) / (sigma2 + d_cov2);
    double sigma_fuse2 = (sigma2 * d_cov2) / (sigma2 + d_cov2);

    depth.ptr<double>(int(pt_ref(1, 0)))[int(pt_ref(0, 0))] = mu_fuse;
    depth_cov2.ptr<double>(int(pt_ref(1, 0)))[int(pt_ref(0, 0))] = sigma_fuse2;

    return true;
}

// The following are too simple so I won’t comment them (actually because I’m lazy)
void plotDepth(const Mat &depth_truth, const Mat &depth_estimate) {

    // imshow("depth_truth", depth_truth * 0.4);
    imwrite("depth_truth.png", depth_truth * 0.4);

    // imshow("depth_estimate", depth_estimate * 0.4);
    imwrite("depth_estimate.png", depth_estimate * 0.4);

    // imshow("depth_error", depth_truth - depth_estimate);
    imwrite("depth_error.png", depth_truth - depth_estimate);

    // waitKey(1);
}

void evaludateDepth(const Mat &depth_truth, const Mat &depth_estimate) {
    double ave_depth_error = 0;     // Average Error
    double ave_depth_error_sq = 0;      // Squared Error
    int cnt_depth_data = 0;
    for (int y = boarder; y < depth_truth.rows - boarder; y++)
        for (int x = boarder; x < depth_truth.cols - boarder; x++) {
            double error = depth_truth.ptr<double>(y)[x] - depth_estimate.ptr<double>(y)[x];
            ave_depth_error += error;
            ave_depth_error_sq += error * error;
            cnt_depth_data++;
        }
    ave_depth_error /= cnt_depth_data;
    ave_depth_error_sq /= cnt_depth_data;

    cout << "Average squared error = " << ave_depth_error_sq << ", average error: " << ave_depth_error << endl;
}

void showEpipolarMatch(const Mat &ref, const Mat &curr, const Vector2d &px_ref, const Vector2d &px_curr) {
    Mat ref_show, curr_show;
    cv::cvtColor(ref, ref_show, cv::COLOR_BGR2RGB);
    cv::cvtColor(curr, curr_show, cv::COLOR_BGR2RGB);

    cv::circle(ref_show, cv::Point2f(px_ref(0, 0), px_ref(1, 0)), 5, cv::Scalar(0, 0, 250), 2);
    cv::circle(curr_show, cv::Point2f(px_curr(0, 0), px_curr(1, 0)), 5, cv::Scalar(0, 0, 250), 2);

    // imshow("ref", ref_show);
    imwrite("ref_1.png", ref_show);

    // imshow("curr", curr_show);
    imwrite("curr_1.png", curr_show);

    // waitKey(1);
}

void showEpipolarLine(const Mat &ref, const Mat &curr, const Vector2d &px_ref, const Vector2d &px_min_curr,
                      const Vector2d &px_max_curr) {

    Mat ref_show, curr_show;
    cv::cvtColor(ref, ref_show, cv::COLOR_BGR2RGB);
    cv::cvtColor(curr, curr_show, cv::COLOR_BGR2RGB);

    cv::circle(ref_show, cv::Point2f(px_ref(0, 0), px_ref(1, 0)), 5, cv::Scalar(0, 255, 0), 2);
    cv::circle(curr_show, cv::Point2f(px_min_curr(0, 0), px_min_curr(1, 0)), 5, cv::Scalar(0, 255, 0), 2);
    cv::circle(curr_show, cv::Point2f(px_max_curr(0, 0), px_max_curr(1, 0)), 5, cv::Scalar(0, 255, 0), 2);
    cv::line(curr_show, Point2f(px_min_curr(0, 0), px_min_curr(1, 0)), Point2f(px_max_curr(0, 0), px_max_curr(1, 0)),
             Scalar(0, 255, 0), 1);

    // imshow("ref", ref_show);
    imwrite("ref_2.png", ref_show);
    // imshow("curr", curr_show);
    imwrite("ref_2.png", ref_show);
    // waitKey(1);
}

Writing dense_mapping.cpp


In [ ]:
!rm -r build
!mkdir build
%cd build

rm: cannot remove 'build': No such file or directory
/content/MyExample_dense/build


In [ ]:
!cmake ..

In [ ]:
!make dense_mapping

In [ ]:
!./dense_mapping

/bin/bash: line 1: ./dense_mapping: No such file or directory


# **ch12/dense_RGBD/pointcloud_mapping.cpp**

## PDF Page 280

In [ ]:
%cd /content/
!rm -r MyExample_pointcloud_mapping
!mkdir MyExample_pointcloud_mapping
%cd /content/MyExample_pointcloud_mapping

/content
rm: cannot remove 'MyExample_pointcloud_mapping': No such file or directory
/content/MyExample_pointcloud_mapping


In [ ]:
!cp -r "/content/drive/MyDrive/slambook_data/ch12/dense_RGBD/data" /content/MyExample_pointcloud_mapping

In [ ]:
%cd data

/content/MyExample_pointcloud_mapping/data


In [ ]:
%%writefile pose.txt

0.000466347 0.00895357 -2.24935 -0.00101358 0.00052453 -0.000231475 0.999999
-0.101611 0.08215 -2.33163 -0.0231916 -0.376659 -0.17448 0.909476
0.310932 -0.432757 -1.48048 0.0492614 0.323821 0.14954 0.932926
-0.0623727 0.225538 -1.07697 -0.0279726 -0.282049 -0.131215 0.949973
-0.0506775 -0.0139318 -0.990509 0.139717 -0.290097 -0.0705922 0.944108

Writing pose.txt


In [ ]:
%cd ..

/content/MyExample_pointcloud_mapping


In [ ]:
%%writefile CMakeLists.txt

cmake_minimum_required(VERSION 3.1)

set(CMAKE_BUILD_TYPE Release)
set(CMAKE_CXX_FLAGS "-std=c++14 -O2")

# opencv
find_package(OpenCV REQUIRED)
include_directories(${OpenCV_INCLUDE_DIRS})

# eigen
#include_directories("/usr/include/eigen3/Eigen")
include_directories("/content/eigen")

# pcl
find_package(PCL REQUIRED)
include_directories(${PCL_INCLUDE_DIRS})
add_definitions(${PCL_DEFINITIONS})
#include_directories("/usr/include/pcl-1.12/pcl")

add_executable(pointcloud_mapping pointcloud_mapping.cpp)
target_link_libraries(pointcloud_mapping ${OpenCV_LIBS} ${PCL_LIBRARIES})

Writing CMakeLists.txt


In [ ]:
%%writefile pointcloud_mapping.cpp

#include <iostream>
#include <fstream>

using namespace std;

#include <opencv2/core/core.hpp>
#include <opencv2/highgui/highgui.hpp>
#include <Eigen/Geometry>
#include <boost/format.hpp>  // for formating strings
#include <pcl/point_types.h>
#include <pcl/io/pcd_io.h>
#include <pcl/filters/voxel_grid.h>
#include <pcl/visualization/pcl_visualizer.h>
#include <pcl/filters/statistical_outlier_removal.h>

int main(int argc, char **argv) {
    vector<cv::Mat> colorImgs, depthImgs;    // Color and depth images
    vector<Eigen::Isometry3d> poses;         // Camera pose

    ifstream fin("/content/MyExample_pointcloud_mapping/data/pose.txt");
    if (!fin) {
        cerr << "cannot find pose file" << endl;
        return 1;
    }

    for (int i = 0; i < 5; i++) {
        boost::format fmt("/content/MyExample_pointcloud_mapping/data/%s/%d.%s"); //Image file formats
        colorImgs.push_back(cv::imread((fmt % "color" % (i + 1) % "png").str()));
        depthImgs.push_back(cv::imread((fmt % "depth" % (i + 1) % "png").str(), -1)); // Use -1 to read the original image

        double data[7] = {0};
        for (int i = 0; i < 7; i++) {
            fin >> data[i];
        }
        Eigen::Quaterniond q(data[6], data[3], data[4], data[5]);
        Eigen::Isometry3d T(q);
        T.pretranslate(Eigen::Vector3d(data[0], data[1], data[2]));
        poses.push_back(T);
    }

    // Calculate point cloud and stitch
    // Camera internal parameters
    double cx = 319.5;
    double cy = 239.5;
    double fx = 481.2;
    double fy = -480.0;
    double depthScale = 5000.0;

    cout << "Converting image to point cloud..." << endl;

    // Define the format used by the point cloud: XYZRGB is used here
    typedef pcl::PointXYZRGB PointT;
    typedef pcl::PointCloud<PointT> PointCloud;

    // Create a new point cloud
    PointCloud::Ptr pointCloud(new PointCloud);
    for (int i = 0; i < 5; i++) {
        PointCloud::Ptr current(new PointCloud);
        cout << "Converting images: " << i + 1 << endl;
        cv::Mat color = colorImgs[i];
        cv::Mat depth = depthImgs[i];
        Eigen::Isometry3d T = poses[i];
        for (int v = 0; v < color.rows; v++)
            for (int u = 0; u < color.cols; u++) {
                unsigned int d = depth.ptr<unsigned short>(v)[u]; // Depth Value
                if (d == 0) continue; // A value of 0 means no measurement
                Eigen::Vector3d point;
                point[2] = double(d) / depthScale;
                point[0] = (u - cx) * point[2] / fx;
                point[1] = (v - cy) * point[2] / fy;
                Eigen::Vector3d pointWorld = T * point;

                PointT p;
                p.x = pointWorld[0];
                p.y = pointWorld[1];
                p.z = pointWorld[2];
                p.b = color.data[v * color.step + u * color.channels()];
                p.g = color.data[v * color.step + u * color.channels() + 1];
                p.r = color.data[v * color.step + u * color.channels() + 2];
                current->points.push_back(p);
            }
        // depth filter and statistical removal
        PointCloud::Ptr tmp(new PointCloud);
        pcl::StatisticalOutlierRemoval<PointT> statistical_filter;
        statistical_filter.setMeanK(50);
        statistical_filter.setStddevMulThresh(1.0);
        statistical_filter.setInputCloud(current);
        statistical_filter.filter(*tmp);
        (*pointCloud) += *tmp;
    }

    pointCloud->is_dense = false;
    cout << "Point cloud sharing" << pointCloud->size() << "Points." << endl;

    // voxel filter
    pcl::VoxelGrid<PointT> voxel_filter;
    double resolution = 0.03;
    voxel_filter.setLeafSize(resolution, resolution, resolution);       // resolution
    PointCloud::Ptr tmp(new PointCloud);
    voxel_filter.setInputCloud(pointCloud);
    voxel_filter.filter(*tmp);
    tmp->swap(*pointCloud);

    cout << "After filtering, the point cloud has" << pointCloud->size() << "Points." << endl;

    pcl::io::savePCDFileBinary("map.pcd", *pointCloud);
    return 0;
}

Writing pointcloud_mapping.cpp


In [ ]:
!rm -r build
!mkdir build
%cd build
!cmake ..

rm: cannot remove 'build': No such file or directory
/content/MyExample_pointcloud_mapping/build
CMake Warning (dev) in CMakeLists.txt:
  No project() command is present.  The top-level CMakeLists.txt file must
  contain a literal, direct call to the project() command.  Add a line of
  code such as

    project(ProjectName)

  near the top of the file, but after cmake_minimum_required().

  CMake is pretending there is a "project(Project)" command on the first
  line.
This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning (dev) in CMakeLists.txt:
  cmake_minimum_required() should be called prior to this top-level project()
  call.  Please see the cmake-commands(7) manual for usage documentation of
  both commands.
This warning is for project developers.  Use -Wno-dev to suppress it.

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Che

In [ ]:
!make

[ 50%] Building CXX object CMakeFiles/pointcloud_mapping.dir/pointcloud_mapping.cpp.o
[100%] Linking CXX executable pointcloud_mapping
[100%] Built target pointcloud_mapping


In [ ]:
!./pointcloud_mapping

正在将图像转换为点云...
转换图像中: 1
转换图像中: 2
转换图像中: 3
转换图像中: 4
转换图像中: 5
点云共有1309800个点.
滤波之后，点云共有31876个点.
